# GOES-19 and GFS over Grenada

Research notebook for Grenada Meteorological Service forecasters: what the GeoNetCast Americas (GNC-A) receiver data can do.

GNC-A rebroadcasts the same NOAA GOES-19 ABI netCDF files that sit in the public AWS bucket `noaa-goes19`. By default this notebook reads them from AWS; set `SOURCE_DIR` to read the receiver's local folder instead.

1. Parameters
2. Find the latest GOES-19 file
3. Band 13 infrared and band 8 water vapour
4. GFS 0.25 degree: pressure, 850 hPa wind, rain

## 1. Parameters

In [ ]:
from pathlib import Path

SOURCE_DIR = None  # e.g. Path("/path/to/receiver/output"); None reads the public AWS bucket
BAND = 13  # ABI channel for the first plot; band 8 (water vapour) is always shown too

LON_MIN, LON_MAX = -66.0, -56.0
LAT_MIN, LAT_MAX = 9.0, 18.0
GRENADA = (-61.75, 12.05)  # lon, lat

## 2. Find the latest GOES-19 file

`ABI-L2-CMIPF` is the full-disk Cloud and Moisture Imagery product. Files are listed per UTC hour; if the current hour has nothing yet, the search steps back one hour at a time.

In [ ]:
import re
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
import s3fs
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import metpy.xarray  # registers the .metpy accessor

BUCKET = "noaa-goes19"
PRODUCT = "ABI-L2-CMIPF"
DATA_DIR = Path("data")  # relative to this notebook: notebooks/satellite/data/, never committed
CACHE = DATA_DIR / "goes19"
CACHE.mkdir(parents=True, exist_ok=True)
fs = s3fs.S3FileSystem(anon=True)


def latest_file(band: int, max_lookback_h: int = 6) -> Path:
    """Newest full-disk CMI file for `band`, from SOURCE_DIR or anonymous AWS."""
    token = f"C{band:02d}_G19"
    if SOURCE_DIR is not None:
        found = sorted(Path(SOURCE_DIR).rglob(f"*{PRODUCT}*{token}*.nc"), key=lambda p: p.name.split("_s")[-1])
        if not found:
            raise FileNotFoundError(f"No {PRODUCT} {token} files under {SOURCE_DIR}")
        return found[-1]
    now = datetime.now(timezone.utc)
    for lag in range(max_lookback_h + 1):
        t = now - timedelta(hours=lag)
        try:
            listing = fs.ls(f"{BUCKET}/{PRODUCT}/{t:%Y}/{t:%j}/{t:%H}/")
        except FileNotFoundError:  # the hour has no files yet
            continue
        keys = sorted(k for k in listing if token in k)
        if keys:
            local = CACHE / Path(keys[-1]).name
            if not local.exists():
                part = local.with_suffix(".part")  # never reuse an interrupted download
                fs.get(keys[-1], str(part))
                part.rename(local)
            return local
    raise FileNotFoundError(f"No {PRODUCT} {token} file in the last {max_lookback_h} h")


path = latest_file(BAND)
path.name

## 3. Infrared and water vapour

The file is on the satellite's fixed-grid (geostationary) projection. MetPy's `parse_cf()` builds the cartopy CRS from the file's metadata; the area box is converted to projection coordinates so only the crop is plotted.

In [ ]:
def plot_band(file: Path, cmap: str, vmin: float, vmax: float, label: str):
    """Plot one CMI file cropped to the area, with coastlines and the scan time in UTC."""
    ds = xr.open_dataset(file).metpy.parse_cf()
    proj = ds["CMI"].metpy.cartopy_crs
    # Corners of the lon/lat box in projection metres give the crop window.
    lons = np.array([LON_MIN, LON_MAX, LON_MIN, LON_MAX])
    lats = np.array([LAT_MIN, LAT_MIN, LAT_MAX, LAT_MAX])
    xs, ys, _ = proj.transform_points(ccrs.PlateCarree(), lons, lats).T
    crop = ds["CMI"].sel(x=slice(xs.min(), xs.max()), y=slice(ys.max(), ys.min()))
    scan = np.datetime_as_string(ds["t"].values, unit="m").replace("T", " ")

    fig = plt.figure(figsize=(8, 6.5))
    ax = fig.add_subplot(projection=proj)
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    img = ax.pcolormesh(crop["x"], crop["y"], crop, cmap=cmap, vmin=vmin, vmax=vmax, transform=proj)
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.right_labels = False
    ax.plot(*GRENADA, marker="o", mfc="none", mec="magenta", markersize=10, transform=ccrs.PlateCarree())
    fig.colorbar(img, ax=ax, shrink=0.8, label=label)
    ax.set_title(f"GOES-19 ABI band {re.search(r'C(\d\d)_G\d\d', file.name).group(1)}  {scan} UTC")
    plt.show()
    ds.close()


# Band 13 clean IR window: brightness temperature in K, cold cloud tops in white.
plot_band(path, cmap="Greys", vmin=-80 + 273.15, vmax=30 + 273.15, label="Brightness temperature (K)")

In [ ]:
# Band 8 upper-level water vapour: cold (moist) air dark blue, warm (dry) air light.
plot_band(latest_file(8), cmap="Blues_r", vmin=200, vmax=270, label="Brightness temperature (K)")

## 4. GFS 0.25 degree

Herbie finds the latest GFS run that is already on the server, then each plot uses that run at 0, 24 and 48 hours. Rain is accumulated since the start of the run, so it has no 0 h panel.

In [ ]:
import pandas as pd
from herbie import Herbie

STEPS = (0, 24, 48)
GFS_DIR = DATA_DIR / "gfs"
GFS_DIR.mkdir(parents=True, exist_ok=True)


def latest_gfs(max_lookback_h: int = 36) -> pd.Timestamp:
    now = pd.Timestamp.now("UTC").tz_localize(None)
    for lag in range(0, max_lookback_h + 1, 6):
        run = (now - pd.Timedelta(hours=lag)).floor("6h")
        try:
            if len(Herbie(run, model="gfs", product="pgrb2.0p25", fxx=max(STEPS), verbose=False, save_dir=GFS_DIR).inventory()):
                return run
        except Exception:
            continue
    raise RuntimeError("No complete GFS run found")


def field(run: pd.Timestamp, fxx: int, search: str) -> xr.DataArray:
    """One GFS field for the area, longitudes as -180..180."""
    H = Herbie(run, model="gfs", product="pgrb2.0p25", fxx=fxx, verbose=False, save_dir=GFS_DIR)
    ds = H.xarray(search, remove_grib=False)
    ds = ds.assign_coords(longitude=((ds.longitude + 180) % 360) - 180).sortby(["longitude", "latitude"])
    return ds.sel(longitude=slice(LON_MIN, LON_MAX), latitude=slice(LAT_MIN, LAT_MAX))


run = latest_gfs()
print("GFS run:", run, "UTC")

In [ ]:
def map_axes(fig, index: int):
    ax = fig.add_subplot(1, len(STEPS), index, projection=ccrs.PlateCarree())
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.plot(*GRENADA, marker="o", mfc="none", mec="magenta", markersize=8)
    return ax


def title(ax, name: str, fxx: int):
    valid = run + pd.Timedelta(hours=fxx)
    ax.set_title(f"{name}\nrun {run:%Y-%m-%d %HZ} +{fxx} h, valid {valid:%d %b %HZ}", fontsize=9)


# Mean sea level pressure (hPa) contours with 850 hPa wind barbs (knots).
fig = plt.figure(figsize=(15, 5))
for i, fxx in enumerate(STEPS, 1):
    ax = map_axes(fig, i)
    msl = field(run, fxx, ":PRMSL:mean sea level")["prmsl"] / 100
    wind = field(run, fxx, ":(?:U|V)GRD:850 mb")
    cs = ax.contour(msl.longitude, msl.latitude, msl, levels=np.arange(1000, 1030, 1), colors="k", linewidths=0.7)
    ax.clabel(cs, fmt="%d", fontsize=7)
    w = wind.isel(longitude=slice(None, None, 4), latitude=slice(None, None, 4))
    ax.barbs(w.longitude, w.latitude, w["u"] * 1.94384, w["v"] * 1.94384, length=5, linewidth=0.5, color="tab:blue")
    title(ax, "MSLP (hPa) and 850 hPa wind (kt)", fxx)
plt.show()

In [ ]:
# Accumulated precipitation since the start of the run (mm).
fig = plt.figure(figsize=(15, 5))
levels = [1, 2, 5, 10, 20, 30, 50, 75, 100]
for i, fxx in enumerate(STEPS, 1):
    ax = map_axes(fig, i)
    if fxx == 0:
        ax.set_title("No accumulation at +0 h", fontsize=9)
        continue
    tp = field(run, fxx, r":APCP:surface:0-")["tp"]
    img = ax.contourf(tp.longitude, tp.latitude, tp, levels=levels, cmap="YlGnBu", extend="max")
    title(ax, "Accumulated precipitation (mm)", fxx)
fig.colorbar(img, ax=fig.axes, shrink=0.8, label="mm since run start")
plt.show()

## 5. Note

These are research outputs, not operational products. Nothing here is a forecast issued by the Grenada Meteorological Service. The scheduled satellite pipeline lives in `scripts/gnc-collector` (the receiver collector).